In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import json
import itertools
from typing import List, Any

import pandas as pd
import torch
from tqdm import tqdm
from transformers import BartTokenizer, BartModel


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "data_dir": "../mall",
            "batch_size": 64,
            "text_columns": ["logs", "RpcName"],
        }
    elif dataset == "train-ticket":
        return {
            "data_dir": "../train-ticket",
            "batch_size": 64,
            "text_columns": ["logs", "RpcName"],
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class TextEmbedder:
    """
    作用：
    1. 读取 data_alignment_2.parquet
    2. 加载 BART tokenizer/model
    3. 对 text_columns 做 embedding
    4. 保存为 data_alignment_3.parquet
    """

    def __init__(self, config: dict):
        self.config = config
        self.data_dir = config["data_dir"]
        self.batch_size = config["batch_size"]
        self.text_columns = config["text_columns"]

        self.input_path = os.path.join(self.data_dir, "data_alignment_2.parquet")
        self.output_path = os.path.join(self.data_dir, "data_alignment_3.parquet")

        self.data_df: pd.DataFrame | None = None
        self.tokenizer = None
        self.model = None

    def load_data(self) -> pd.DataFrame:
        self.data_df = pd.read_parquet(self.input_path)
        return self.data_df

    def load_model(self):
        """
        与原代码一致：加载 facebook/bart-large
        """
        self.tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
        self.model = BartModel.from_pretrained("facebook/bart-large")

        if torch.cuda.is_available():
            self.model.cuda()

        self.model.eval()
        return self.tokenizer, self.model

    def encode_texts_set(self, texts: List[str]) -> List[List[float]]:
        """
        与原逻辑一致：
        1. 先对文本去重
        2. 只对唯一文本做编码
        3. 再映射回原顺序

        输出：
        List[List[float]]
        """
        if self.tokenizer is None or self.model is None:
            self.load_model()

        texts_set = list(set(texts))
        text_to_id = {text: idx for idx, text in enumerate(texts_set)}
        texts_id = [text_to_id[text] for text in texts]

        batches = [
            texts_set[i:i + self.batch_size]
            for i in range(0, len(texts_set), self.batch_size)
        ]

        encoded_vectors = []
        for batch in tqdm(batches, postfix="encode text batch"):
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            )

            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs)

            batch_vectors = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
            encoded_vectors.extend(batch_vectors)

        embeds = [
            [float(item) for item in encoded_vectors[idx]]
            for idx in texts_id
        ]
        return embeds

    @staticmethod
    def split_by_lengths(original_list: List[Any], lengths: List[int]) -> List[List[Any]]:
        """
        按长度列表把展平后的向量重新分组。
        """
        it = iter(original_list)
        return [list(itertools.islice(it, length)) for length in lengths]

    def encode_scalar_text_column(self, column_name: str) -> None:
        """
        编码标量文本列，如 RpcName
        原始：
            "serviceA: op1"
        输出：
            [0.12, -0.03, ...]
        """
        if self.data_df is None:
            self.load_data()

        text_list = self.data_df[column_name].tolist()
        self.data_df[column_name] = self.encode_texts_set(text_list)

    def encode_list_text_column(self, column_name: str) -> None:
        """
        编码列表文本列，如 logs
        原始：
            ["log1", "log2"]
        输出：
            [[...embedding1...], [...embedding2...]]
        """
        if self.data_df is None:
            self.load_data()

        text_list = [item for item in self.data_df[column_name].to_list()]
        lengths = [len(text) for text in text_list]

        text_flatten = []
        for text_group in text_list:
            text_flatten.extend(text_group)

        encoded_text_flatten = self.encode_texts_set(text_flatten)
        self.data_df[column_name] = self.split_by_lengths(encoded_text_flatten, lengths)

    # 将每个span的服务接口名和其每条日志编码为语义向量
    def process_column(self, column_name: str) -> None:
        """
        判断列类型：
        - 如果第一项是 str，则按标量文本列处理
        - 否则按 list[str] 文本列处理
        """
        if self.data_df is None:
            self.load_data()

        first_element = self.data_df[column_name].iloc[0]

        if isinstance(first_element, str):
            self.encode_scalar_text_column(column_name)
        else:
            self.encode_list_text_column(column_name)

    def save(self) -> str:
        if self.data_df is None:
            raise ValueError("No dataframe to save.")
        # 得到一个宽表，每一行是一个span对应的数据,将服务接口名RpcName和日志模板用Bert编码为语义向量
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |
        self.data_df.to_parquet(self.output_path, index=False)
        print(f"save embedded text data to: {self.output_path}")
        return self.output_path

    def run(self) -> pd.DataFrame:
        # 得到一个宽表，每一行是一个span对应的数据,日志解析为模板
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |
        self.load_data()
        self.load_model()

        # ["logs", "RpcName"]对横表的这两列进行处理
        for column_name in self.text_columns:
            self.process_column(column_name)

        self.save()
        return self.data_df

/home/lenovo/miniconda3/envs/Medicine/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# dataset = "mall"   # "mall" 或 "train-ticket"
for dataset in ["mall", "train-ticket"]:
    config = get_config(dataset)
    embedder = TextEmbedder(config)
    result_df = embedder.run()
    print("\nembedded_df shape:", result_df.shape)


/home/lenovo/miniconda3/envs/Medicine/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00, 19.99it/s, encode text batch]


save embedded text data to: ../mall/data_alignment_3.parquet

embedded_df shape: (1010669, 53)


100%|██████████| 9/9 [00:00<00:00,  9.50it/s, encode text batch]


save embedded text data to: ../train-ticket/data_alignment_3.parquet

embedded_df shape: (38240, 51)


In [8]:
import os

print(os.path.exists("facebook"))
print(os.path.exists("facebook/bart-large"))

from transformers.utils import cached_file

False
False
